<a href="https://colab.research.google.com/github/Seifeddin84/SISCOIN/blob/Gemini_tunable_pinn_better_physics/LAST_PINN_TUNABLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wfdb

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 136.0 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
Mounted at /content/drive


In [10]:
"""
====================================================================================
VERSION B: REFACTORED (Tunable Weights & Inits)
====================================================================================
Epochs: 3000 Adam + 300 L-BFGS
Target R²: 0.92
Corrections:
1. Loss weights are in the CONFIG block.
2. Initial k and c values are now in the CONFIG block.
====================================================================================
"""

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import wfdb
from scipy.signal import butter, filtfilt
from sklearn.metrics import r2_score
import warnings
import time
from datetime import datetime
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
BASE_DATA_PATH = "/content/drive/MyDrive/human-balance-evaluation-database-1.0.0"
METADATA_FILE_PATH = os.path.join(BASE_DATA_PATH, 'BDSinfo.txt')
RESULTS_DIR = f'PINN_Results_VB_Optimal/'
PLOTS_DIR = os.path.join(RESULTS_DIR, 'plots/')
AGGREGATE_DIR = os.path.join(RESULTS_DIR, 'results/')

# Training settings
MODEL_TYPE = 'inverted_pendulum'
ADAM_EPOCHS = 3000
LBFGS_EPOCHS = 300
LEARNING_RATE = 1e-4
GRAD_CLIP_MAX_NORM = 1.0

# Optimization flags
SAVE_PLOTS_FREQUENCY = 1      # Save every Nth plot
LBFGS_MAX_ITER = 10           # Reduced from 20

# --- Loss function weights ---
W_DATA = 100.0      # Priority on fitting the data
W_PHYS = 10.0       # "Sweet spot" physics weight
W_IC_POS = 100.0    # Priority on matching start position
W_IC_VEL = 1.0      # Lower priority on start velocity

# --- NEW: Initialization settings ---
INIT_K_MULTIPLIER = 1.05  # Initial k = m*g*l * THIS_VALUE
INIT_C_VALUE = 100.0      # Initial c = THIS_VALUE
# --- End of new section ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*80)
print(f"VERSION B: REFACTORED (Tunable Weights & Inits)")
print("="*80)
print(f"Device: {device}")
print(f"Epochs: {ADAM_EPOCHS} Adam + {LBFGS_EPOCHS} L-BFGS")
print(f"Weights: Data={W_DATA}, Physics={W_PHYS}, IC_Pos={W_IC_POS}, IC_Vel={W_IC_VEL}")
print(f"Inits: k_mult={INIT_K_MULTIPLIER}, c_val={INIT_C_VALUE}")
print(f"Target R²: 0.92")
print("="*80 + "\n")

for directory in [RESULTS_DIR, PLOTS_DIR, AGGREGATE_DIR]:
    os.makedirs(directory, exist_ok=True)

# ============================================================================
# METADATA HANDLING
# ============================================================================
def map_metadata_columns(df):
    """Auto-detect column names"""
    mapping = {}
    cols = {c.lower(): c for c in df.columns}

    searches = {
        'trial_name': ['trial', 'trial_id'],
        'subject_id': ['subject', 'subject_id'],
        'age': ['age'],
        'gender': ['gender', 'sex'],
        'height': ['height'],
        'mass': ['weight', 'mass'],
        'surface': ['surface'],
        'eyes': ['eyes', 'vision'],
        'trial_num': ['trial_num', 'rep']
    }

    for key, variants in searches.items():
        for v in variants:
            if v in cols:
                mapping[key] = cols[v]
                break

    return mapping

def get_meta(row, mapping, key, default=None):
    """Safe metadata extraction"""
    if key in mapping:
        try:
            val = row[mapping[key]].values[0]
            return default if pd.isna(val) else val
        except:
            return default
    return default

# ============================================================================
# DATA LOADING
# ============================================================================
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)

def load_trial_data(trial_name):
    """Fast data loading with minimal overhead"""
    try:
        rec = wfdb.rdrecord(os.path.join(BASE_DATA_PATH, trial_name))

        if 'COPx' not in rec.sig_name or 'COPy' not in rec.sig_name:
            print(f"Skipping {trial_name}: Missing 'COPx' or 'COPy'.")
            return None, None, None, None

        x = rec.p_signal[:, rec.sig_name.index('COPx')]
        y = rec.p_signal[:, rec.sig_name.index('COPy')]

        if not np.isfinite(x).all() or not np.isfinite(y).all():
            print(f"Skipping {trial_name}: Contains NaN/Inf values.")
            return None, None, None, None

        fs = rec.fs
        t = np.linspace(0, (len(x) - 1) / fs, len(x))

        # Filter and normalize
        x_filt = butter_lowpass_filter(x, 2.0, fs)
        y_filt = butter_lowpass_filter(y, 2.0, fs)
        x_norm = (x_filt - np.mean(x_filt)) / 100.0 #converts cm to m
        y_norm = (y_filt - np.mean(y_filt)) / 100.0 #converts cm to m

        # Convert to tensors
        t_t = torch.tensor(t, dtype=torch.float32).view(-1, 1)
        x_t = torch.tensor(x_norm, dtype=torch.float32).view(-1, 1)
        y_t = torch.tensor(y_norm, dtype=torch.float32).view(-1, 1)

        return t_t, x_t, y_t, t.max()

    except Exception as e:
        print(f"ERROR loading {trial_name}: {e}")
        return None, None, None, None
# ============================================================================
# NEURAL NETWORK ARCHITECTURE
# ============================================================================
class FourierFeatures(nn.Module):
    def __init__(self, input_dim, mapping_size, scale):
        super().__init__()
        self.B = nn.Parameter(
            torch.randn(input_dim, mapping_size) * scale,
            requires_grad=False
        )

    def forward(self, x):
        x_proj = 2 * np.pi * x @ self.B
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_layers, hidden_units):
        super().__init__()
        self.fourier = FourierFeatures(input_dim, 64, 10.0)

        layers = [nn.Linear(128, hidden_units), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(hidden_units, hidden_units), nn.Tanh()])
        layers.append(nn.Linear(hidden_units, output_dim))

        self.network = nn.Sequential(*layers)
        self.network.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                m.bias.data.fill_(0.0)

    def forward(self, t):
        return self.network(self.fourier(t))

class BalancePINN(nn.Module):
    # --- CHANGE 1: Modified __init__ to accept init_k_val and init_c_val ---
    def __init__(self, t_max, m, l, weights, init_k_val, init_c_val, g=9.81):
        super().__init__()
        self.net = MLP(1, 2, 8, 64)
        self.m, self.l, self.g, self.t_max = m, l, g, t_max

        # --- CHANGE 2: Use passed-in initial values ---
        init_k = torch.tensor(init_k_val, device=device, dtype=torch.float32)
        init_c = torch.tensor(init_c_val, device=device, dtype=torch.float32)

        self.log_k_x = nn.Parameter(torch.log(init_k.clone()))
        self.log_c_x = nn.Parameter(torch.log(init_c.clone()))
        self.log_k_y = nn.Parameter(torch.log(init_k.clone()))
        self.log_c_y = nn.Parameter(torch.log(init_c.clone()))

        # Use weights from the dict
        self.w_data = torch.tensor(weights['data'], device=device, requires_grad=False)
        self.w_ic_pos = torch.tensor(weights['ic_pos'], device=device, requires_grad=False)
        self.w_ic_vel = torch.tensor(weights['ic_vel'], device=device, requires_grad=False)
        self.w_phys = torch.tensor(weights['phys'], device=device, requires_grad=False)

        self.mse = nn.MSELoss()
        self.huber = nn.SmoothL1Loss(beta=0.1)

    def forward(self, t_norm):
        return self.net(t_norm)

    def physics_residual(self, t_norm):
        t_norm.requires_grad_(True)
        xy = self.forward(t_norm)
        N, M = xy[:, 0:1], xy[:, 1:2]

        N_t = torch.autograd.grad(N, t_norm, torch.ones_like(N), create_graph=True)[0]
        M_t = torch.autograd.grad(M, t_norm, torch.ones_like(M), create_graph=True)[0]
        N_tt = torch.autograd.grad(N_t, t_norm, torch.ones_like(N_t), create_graph=True)[0]
        M_tt = torch.autograd.grad(M_t, t_norm, torch.ones_like(M_t), create_graph=True)[0]

        x_t, x_tt = N_t / self.t_max, N_tt / (self.t_max ** 2)
        y_t, y_tt = M_t / self.t_max, M_tt / (self.t_max ** 2)

        k_x, c_x = torch.exp(self.log_k_x), torch.exp(self.log_c_x)
        k_y, c_y = torch.exp(self.log_k_y), torch.exp(self.log_c_y)

        c_x_n = c_x / (self.m * self.l ** 2)
        k_x_n = (k_x / (self.m * self.l ** 2)) - (self.g / self.l)
        c_y_n = c_y / (self.m * self.l ** 2)
        k_y_n = (k_y / (self.m * self.l ** 2)) - (self.g / self.l)

        f_x = x_tt + c_x_n * x_t + k_x_n * N
        f_y = y_tt + c_y_n * y_t + k_y_n * M

        return f_x, f_y

    def compute_loss(self, t_data, x_data, y_data):
        t_norm = t_data / self.t_max

        # Data loss
        pred = self.forward(t_norm)
        loss_data = self.mse(pred[:, 0:1], x_data) + self.mse(pred[:, 1:2], y_data)

        # IC position
        t0 = t_norm[0:1].clone().requires_grad_(True)
        xy0 = self.forward(t0)
        loss_ic_pos = self.mse(xy0[:, 0:1], x_data[0:1]) + self.mse(xy0[:, 1:2], y_data[0:1])

        # IC velocity
        t0v = t_norm[0:1].clone().requires_grad_(True)
        xy0v = self.forward(t0v)
        vx = torch.autograd.grad(xy0v[:, 0:1], t0v, torch.ones_like(xy0v[:, 0:1]), create_graph=True)[0]
        vy = torch.autograd.grad(xy0v[:, 1:2], t0v, torch.ones_like(xy0v[:, 1:2]), create_graph=True)[0]

        vx_pred, vy_pred = vx / self.t_max, vy / self.t_max
        vx_data = (x_data[1:2] - x_data[0:1]) / 0.01
        vy_data = (y_data[1:2] - y_data[0:1]) / 0.01

        loss_ic_vel = self.huber(vx_pred, vx_data) + self.huber(vy_pred, vy_data)

        # Physics
        t_phys = t_norm.clone().requires_grad_(True)
        fx, fy = self.physics_residual(t_phys)
        loss_phys = self.huber(fx, torch.zeros_like(fx)) + self.huber(fy, torch.zeros_like(fy))

        loss = (self.w_data * loss_data + self.w_ic_pos * loss_ic_pos +
                self.w_ic_vel * loss_ic_vel + self.w_phys * loss_phys)

        return loss, loss_data, loss_phys, loss_ic_pos + loss_ic_vel

# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_single_trial(trial_name, metadata_row, col_map, trial_idx):
    """Optimized training with minimal overhead"""

    t_data, x_data, y_data, t_max = load_trial_data(trial_name)
    if t_data is None:
        return None

    t_data = t_data.to(device)
    x_data = x_data.to(device)
    y_data = y_data.to(device)

    m = get_meta(metadata_row, col_map, 'mass', 70.0)
    h = get_meta(metadata_row, col_map, 'height', 170.0)
    l = h / 100.0 if h else 1.7

    # Collect weights
    loss_weights = {
        'data': W_DATA,
        'phys': W_PHYS,
        'ic_pos': W_IC_POS,
        'ic_vel': W_IC_VEL
    }

    # --- CHANGE 3: Calculate inits based on config and pass them to PINN ---
    init_k_val = m * 9.81 * l * INIT_K_MULTIPLIER
    init_c_val = INIT_C_VALUE

    pinn = BalancePINN(t_max, m, l, loss_weights, init_k_val, init_c_val).to(device)

    # Adam phase with aggressive LR schedule
    opt_adam = optim.Adam(pinn.parameters(), lr=LEARNING_RATE)
    sched = optim.lr_scheduler.StepLR(opt_adam, step_size=1000, gamma=0.5)

    pinn.train()
    for _ in range(ADAM_EPOCHS):
        opt_adam.zero_grad()
        loss, _, _, _ = pinn.compute_loss(t_data, x_data, y_data)
        if torch.isnan(loss):
            return None
        loss.backward()
        nn.utils.clip_grad_norm_(pinn.parameters(), GRAD_CLIP_MAX_NORM)
        opt_adam.step()
        sched.step()

    # L-BFGS phase
    opt_lbfgs = optim.LBFGS(
        filter(lambda p: p.requires_grad, pinn.parameters()),
        lr=0.1, max_iter=LBFGS_MAX_ITER, line_search_fn="strong_wolfe"
    )

    def closure():
        with torch.enable_grad():
            opt_lbfgs.zero_grad()
            loss, _, _, _ = pinn.compute_loss(t_data, x_data, y_data)
            if torch.isnan(loss):
                raise RuntimeError("NaN")
            loss.backward()
        return loss

    for _ in range(LBFGS_EPOCHS):
        try:
            opt_lbfgs.step(closure)
        except:
            break

    # Evaluation
    pinn.eval()

    # Moved loss calculation *outside* of the no_grad() block
    final_loss, _, _, _ = pinn.compute_loss(t_data, x_data, y_data)

    with torch.no_grad():
        k_x = torch.exp(pinn.log_k_x).item()
        c_x = torch.exp(pinn.log_c_x).item()
        k_y = torch.exp(pinn.log_k_y).item()
        c_y = torch.exp(pinn.log_c_y).item()

        t_norm = t_data / t_max
        pred = pinn.forward(t_norm)
        x_pred = pred[:, 0:1].cpu().numpy()
        y_pred = pred[:, 1:2].cpu().numpy()

    x_act = x_data.cpu().numpy()
    y_act = y_data.cpu().numpy()

    r2_x = r2_score(x_act, x_pred)
    r2_y = r2_score(y_act, y_pred)
    rmse_x = np.sqrt(np.mean((x_act - x_pred) ** 2))
    rmse_y = np.sqrt(np.mean((y_act - y_pred) ** 2))

    cop_path = np.sum(np.sqrt(np.diff(x_act.flatten()) ** 2 + np.diff(y_act.flatten()) ** 2))
    cop_area = np.pi * np.std(x_act) * np.std(y_act)

    # Save occasional plots
    if trial_idx % SAVE_PLOTS_FREQUENCY == 0:
        try:
            save_plot(t_data.cpu().numpy(), x_act, y_act, x_pred, y_pred, trial_name, r2_x, r2_y)
        except:
            pass

    return {
        'trial_name': trial_name,
        'k_x': k_x, 'c_x': c_x, 'k_y': k_y, 'c_y': c_y,
        'final_loss': final_loss.item(),
        'r2_x': r2_x, 'r2_y': r2_y,
        'rmse_x': rmse_x, 'rmse_y': rmse_y,
        'cop_path_length': cop_path,
        'cop_area': cop_area,
        'cop_velocity': cop_path / t_max,
        'mass': m, 'height': l,
        'subject_id': get_meta(metadata_row, col_map, 'subject_id', 'Unknown'),
        'age': get_meta(metadata_row, col_map, 'age', np.nan),
        'gender': get_meta(metadata_row, col_map, 'gender', 'Unknown'),
        'surface': get_meta(metadata_row, col_map, 'surface', 'Unknown'),
        'eyes': get_meta(metadata_row, col_map, 'eyes', 'Unknown'),
        'trial_num': get_meta(metadata_row, col_map, 'trial_num', 1),
    }

def save_plot(t, x_act, y_act, x_pred, y_pred, trial_name, r2_x, r2_y):
    """Minimal fast plotting"""
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    ax[0].plot(t, x_act, 'b-', alpha=0.5, lw=0.5)
    ax[0].plot(t, x_pred, 'r-', lw=1)
    ax[0].set_title(f'X (R²={r2_x:.3f})')
    ax[1].plot(t, y_act, 'b-', alpha=0.5, lw=0.5)
    ax[1].plot(t, y_pred, 'r-', lw=1)
    ax[1].set_title(f'Y (R²={r2_y:.3f})')
    ax[2].plot(x_act, y_act, 'b-', alpha=0.3, lw=0.5)
    ax[2].plot(x_pred, y_pred, 'r-', lw=0.5)
    ax[2].set_title('Stabilogram')
    ax[2].axis('equal')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f"{trial_name}.png"), dpi=75)
    plt.close()

# ============================================================================
# MAIN TRAINING LOOP
# ============================================================================
def run_training(metadata_df, col_map):
    """Main training loop with progress tracking"""

    trial_col = col_map.get('trial_name', 'Trial')
    trials = metadata_df[trial_col].values
    trials = trials[:1] # Still set to 1 trial for testing

    print(f"Starting training on {len(trials)} trials (TEST RUN)...\n")
    start_time = time.time()

    all_results = []

    for idx, trial_name in enumerate(tqdm(trials, desc="Training")):
        trial_meta = metadata_df[metadata_df[trial_col] == trial_name]

        # REMOVED the try/except block to see all errors!
        result = train_single_trial(trial_name, trial_meta, col_map, idx)
        if result:
            all_results.append(result)
        else:
            print(f"Trial {trial_name} failed.")

        # Progress updates
        if (idx + 1) % 100 == 0:
            elapsed = time.time() - start_time
            rate = (idx + 1) / elapsed
            remaining = (len(trials) - idx - 1) / rate
            print(f"\n[{idx+1}/{len(trials)}] " +
                  f"Processed: {len(all_results)} | " +
                  f"ETA: {remaining/3600:.1f}h")

    # Save results
    results_df = pd.DataFrame(all_results)

    if results_df.empty:
        print("\n" + "="*80)
        print("TRAINING FAILED TO PRODUCE ANY RESULTS.")
        print("="*80)
        return None

    csv_path = os.path.join(AGGREGATE_DIR, f'results_versionB_Refactored.csv')
    results_df.to_csv(csv_path, index=False)

    elapsed_total = time.time() - start_time

    print("\n" + "="*80)
    print("FULL TRAINING COMPLETE!")
    print("="*80)
    print(f"Total time: {elapsed_total/3600:.2f} hours")
    print(f"Successful: {len(results_df)} / {len(trials)} trials")
    print(f"Results: {csv_path}")
    print("\nQuality Metrics:")
    print(f"  Mean R²(X): {results_df['r2_x'].mean():.4f} ± {results_df['r2_x'].std():.4f}")
    print(f"  Mean R²(Y): {results_df['r2_y'].mean():.4f} ± {results_df['r2_y'].std():.4f}")
    print(f"  Overall R²: {results_df[['r2_x', 'r2_y']].mean().mean():.4f}")
    print(f"  Trials R²≥0.92: {((results_df['r2_x']>=0.92) & (results_df['r2_y']>=0.92)).sum()} / {len(results_df)}")
    print(f"  Trials R²≥0.90: {((results_df['r2_x']>=0.90) & (results_df['r2_y']>=0.90)).sum()} / {len(results_df)}")
    print("="*80)

    return results_df

# ============================================================================
# MAIN
# ============================================================================
def main():
    metadata_df = pd.read_csv(METADATA_FILE_PATH, sep='\t')
    print(f"Loaded metadata: {len(metadata_df)} trials\n")

    col_map = map_metadata_columns(metadata_df)
    print("Column mapping complete\n")

    results_df = run_training(metadata_df, col_map)

    return results_df

if __name__ == "__main__":
    main()

VERSION B: REFACTORED (Tunable Weights & Inits)
Device: cuda
Epochs: 3000 Adam + 300 L-BFGS
Weights: Data=100.0, Physics=10.0, IC_Pos=100.0, IC_Vel=1.0
Inits: k_mult=1.05, c_val=100.0
Target R²: 0.92

Loaded metadata: 1930 trials

Column mapping complete

Starting training on 1 trials (TEST RUN)...



Training: 100%|██████████| 1/1 [01:43<00:00, 103.73s/it]


FULL TRAINING COMPLETE!
Total time: 0.03 hours
Successful: 1 / 1 trials
Results: PINN_Results_VB_Optimal/results/results_versionB_Refactored.csv

Quality Metrics:
  Mean R²(X): 0.7838 ± nan
  Mean R²(Y): 0.9291 ± nan
  Overall R²: 0.8565
  Trials R²≥0.92: 0 / 1
  Trials R²≥0.90: 0 / 1
